# Transformer lag configuration study

In [ ]:
import os
import sys
import json
import hashlib
import time
import gc
import warnings
from pathlib import Path
from typing import Dict, Any, List, Tuple, Union

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline
from models import ProbabilisticTransformer
from core.trainer import Trainer
from core.evaluator import Evaluator
from transformations import StandardScalingTransformation

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# GPU Setup
try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-02-15 16:12:55.214281: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771168375.231993 1421584 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771168375.237089 1421584 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771168375.250740 1421584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771168375.250757 1421584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771168375.250759 1421584 computation_placer.cc:177] computation placer alr

GPUs detected: 1


## Configuration

In [ ]:
# Results directory
RESULTS_DIR = Path(project_root) / "results" / "lag_study"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "lag_study_results.json"

# Base model parameters
BASE_MODEL_CONFIG = {
    "d_model": 224,      
    "num_heads": 7,
    "num_layers": 3,
    "ff_dim": 448,
    "dropout": 0.15,
}

# Training settings
TRAIN_SETTINGS = {
    "batch_size": 32,
    "epochs": 30,
    "learning_rate": 7e-4,
    "patience": 7,
    "validation_split": 0.1,
}

OUTPUT_HORIZON = 24
N_RUNS = 3

## Lag generation logic

Functions to generate indices and create dataset sequences

In [ ]:
def get_lag_indices(config_name: str) -> List[int]:
    # Parse configuration name and returns a list of lag indices

    import re
    lags = set()
    
    # Recent windows (e.g. recent_24h)
    if "recent_" in config_name:
        match = re.search(r"recent_(\d+)h", config_name)
        if match:
            hours = int(match.group(1))
            for h in range(1, hours + 1):
                lags.add(h)
    
    # Daily (e.g. daily_same_hour_90d)
    if "daily_same_hour_" in config_name:
        match = re.search(r"daily_same_hour_(\d+)d", config_name)
        if match:
            days = int(match.group(1))
            for d in range(1, days + 1):
                lags.add(d * 24)

    # Weekly (e.g. weekly_same_hour_26w)
    if "weekly_same_hour_" in config_name:
        match = re.search(r"weekly_same_hour_(\d+)w", config_name)
        if match:
            weeks = int(match.group(1))
            for w in range(1, weeks + 1):
                lags.add(w * 168)

    # Weekly anchors
    if "weekly" in config_name and "weekly_same_hour" not in config_name:
        match = re.search(r"weekly(\d+)", config_name)
        if match:
            weeks = int(match.group(1))
            for w in range(1, weeks + 1):
                lags.add(w * 168)
    
    # Daily anchors
    if "daily" in config_name and "daily_same_hour" not in config_name:
        match = re.search(r"daily(\d+)", config_name)
        if match:
            days = int(match.group(1))
            for d in range(1, days + 1):
                lags.add(d * 24)

    if not lags:
        raise ValueError(f"Could not get lags from config: {config_name}")
        
    return sorted(list(lags), reverse=True)

def create_lagged_dataset(df: pd.DataFrame, lag_indices: List[int], output_horizon: int) -> Tuple[np.ndarray, np.ndarray]:
    #Creates X (samples, n_lags, features) and y (samples, horizon) dataset

    data = df.values
    n_features = data.shape[1]
    max_lag = max(lag_indices)
    n_samples = len(data) - max_lag - output_horizon + 1
    
    if n_samples <= 0:
        return np.empty((0, len(lag_indices), data.shape[1])), np.empty((0, output_horizon))
    
    t_indices = np.arange(max_lag, len(data) - output_horizon + 1)
    
    lag_arr = np.array(lag_indices)
    X_idx = t_indices[:, None] - lag_arr[None, :]
    
    X = data[X_idx]
    
    y_offsets = np.arange(output_horizon)
    y_idx = t_indices[:, None] + y_offsets[None, :]
    
    y = data[y_idx, 0]
    
    
    valid_mask = ~(np.isnan(X).any(axis=(1,2)) | np.isnan(y).any(axis=1))
    X = X[valid_mask]
    y = y[valid_mask]
    
    return X, y

## Experiment configurations

In [ ]:
EXPERIMENTS = []

# Recent windows
for h in [3, 24, 72, 168, 336]:
    EXPERIMENTS.append(f"recent_{h}h")

# Sparse seasonal anchors
EXPERIMENTS.append("daily_same_hour_90d")
EXPERIMENTS.append("daily_same_hour_200d")
EXPERIMENTS.append("weekly_same_hour_26w")
EXPERIMENTS.append("weekly_same_hour_52w")

# Hybrid configurations
EXPERIMENTS.append("recent_24h_plus_daily14")
EXPERIMENTS.append("recent_24h_plus_weekly4")

# 4. Grid search sweep
recent_sweep = [36, 48, 60, 72, 84, 96]
weekly_sweep = [4, 8, 12]

for r in recent_sweep:
    for w in weekly_sweep:
        EXPERIMENTS.append(f"recent_{r}h_plus_weekly{w}")

print(f"Total experiments defined: {len(EXPERIMENTS)}")
print("Sample experiments:", EXPERIMENTS[:5])

Total experiments defined: 29
Sample experiments: ['recent_3h', 'recent_24h', 'recent_72h', 'recent_168h', 'recent_336h']


## Caching

In [5]:
def load_cache() -> Dict[str, Any]:
    if CACHE_FILE.exists():
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except json.JSONDecodeError:
            print("Cache file corrupted, starting fresh.")
            return {}
    return {}

def save_cache(cache: Dict[str, Any]) -> None:
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)

## Main experiment loop

In [ ]:
# Load data once
print("Loading base data")
data_config = DataConfig(dataset_name="BE_ENTSOE")
pipeline = DataPipeline(data_config)
train_df_raw, val_df_raw, test_df_raw = pipeline.get_data_splits()

# Drop columns with NaN values
cols_with_nan = train_df_raw.columns[train_df_raw.isna().any()].tolist()
if cols_with_nan:
    print(f"Dropping {len(cols_with_nan)} columns with NaN: {cols_with_nan}")
    train_df_raw = train_df_raw.drop(columns=cols_with_nan)
    val_df_raw = val_df_raw.drop(columns=cols_with_nan)
    test_df_raw = test_df_raw.drop(columns=cols_with_nan)

print(f"Features used: {list(train_df_raw.columns)}")

print(f"Train samples: {len(train_df_raw)}")
print(f"Val samples: {len(val_df_raw)}")
print(f"Test samples: {len(test_df_raw)}")

cache = load_cache()

for i, exp_name in enumerate(EXPERIMENTS):
    print(f"\n[{i+1}/{len(EXPERIMENTS)}] Processing: {exp_name}")
    
    if exp_name in cache:
        print("  -> Found in cache. Skipping.")
        continue
    
    # Parse lags
    try:
        lags = get_lag_indices(exp_name)
    except ValueError as e:
        print(f"  -> Error parsing config: {e}")
        continue
        
    n_lags = len(lags)
    print(f"  -> Lags: {n_lags} total (Max lag: {max(lags)}) ")
    
    # Prepare data
    print("  -> Creating matrices...")
    X_train, y_train = create_lagged_dataset(train_df_raw, lags, OUTPUT_HORIZON)
    X_val, y_val = create_lagged_dataset(val_df_raw, lags, OUTPUT_HORIZON)
    X_test, y_test = create_lagged_dataset(test_df_raw, lags, OUTPUT_HORIZON)
    
    if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
        print(f"  -> Skipping: insufficient data (train={len(X_train)}, val={len(X_val)}, test={len(X_test)})")
        continue
    
    # Standard scaling
    transform = StandardScalingTransformation()
    transform.fit(X_train, y_train)
    
    X_train_s, y_train_s = transform.transform(X_train, y_train)
    X_val_s, y_val_s = transform.transform(X_val, y_val)
    X_test_s, y_test_s = transform.transform(X_test, y_test)
    
    # Run repeats
    run_metrics = []
    run_times = []
    
    for run_idx in range(N_RUNS):
        print(f"    Run {run_idx+1}/{N_RUNS}...")
        
        # Cleanup
        tf.keras.backend.clear_session()
        gc.collect()
        
        # Configs
        current_data_config = DataConfig(
            dataset_name="BE_ENTSOE",
            input_window=n_lags, 
            output_horizon=OUTPUT_HORIZON
        )
        
        model_conf = ModelConfig(**BASE_MODEL_CONFIG)
        train_conf = TrainingConfig(**TRAIN_SETTINGS)
        
        exp_conf = ExperimentConfig(
            name=f"{exp_name}_run{run_idx}",
            data_config=current_data_config,
            model_config=model_conf,
            training_config=train_conf,
            head_type="johnson_su"
        )
        
        # Initialize & Train
        model = ProbabilisticTransformer(exp_conf)
        trainer = Trainer(exp_conf)
        
        start_time = time.time()
        history = trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s)
        fit_time = time.time() - start_time
        
        # Evaluate
        evaluator = Evaluator(model, transform)
        metrics = evaluator.evaluate(X_test_s, y_test) # user raw y_test for metrics
        
        run_metrics.append(metrics)
        run_times.append(fit_time)
        
        del model, trainer, evaluator
    
    # Aggregate results
    avg_metrics = {}
    std_metrics = {}
    
    keys = run_metrics[0].keys()
    for k in keys:
        values = [m[k] for m in run_metrics]
        avg_metrics[k] = float(np.mean(values))
        std_metrics[k] = float(np.std(values))
    
    result_entry = {
        "experiment": exp_name,
        "lags_count": n_lags,
        "metrics_avg": avg_metrics,
        "metrics_std": std_metrics,
        "avg_fit_time": float(np.mean(run_times)),
        "runs": N_RUNS
    }
    
    # Save update
    cache[exp_name] = result_entry
    save_cache(cache)
    
    print(f"  -> Done. Avg MAE: {avg_metrics.get('MAE', 0):.4f}")

Loading base data...
Dropping 5 columns with NaN: ['FR_Generation_Forecast', 'BE_Wind_Forecast', 'BE_Solar_Forecast', 'Flow_BE_NL', 'BE_Generation_Actual']
Features used: ['Prices', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'DayOfYear', 'WeekOfYear', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'NL_Prices', 'FR_Prices', 'FR_Load_Forecast', 'BE_Load_Actual', 'BE_Load_Forecast', 'Price_Spread_FR', 'Flow_BE_FR', 'BE_Load_Imbalance', 'Price_Spread_NL', 'Flow_BE_DE']
Train samples: 19700
Val samples: 2188
Test samples: 13278

[1/29] Processing: recent_3h
  -> Found in cache. Skipping.

[2/29] Processing: recent_24h
  -> Found in cache. Skipping.

[3/29] Processing: recent_72h
  -> Found in cache. Skipping.

[4/29] Processing: recent_168h
  -> Found in cache. Skipping.

[5/29] Processing: recent_336h
  -> Found in cache. Skipping.

[6/29] Processing: daily_same_hour_90d
  -> Found in cache. Skipping.

[7/29] Processing: daily_same_hour_200d
  -> Lags: 

I0000 00:00:1771168379.367054 1421584 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7483 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:01:00.0, compute capability: 6.1


Epoch 1/30


I0000 00:00:1771168385.336323 1422744 service.cc:152] XLA service 0x727d7c004ce0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1771168385.336337 1422744 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1080, Compute Capability 6.1
2026-02-15 16:13:05.526616: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1771168386.519799 1422744 cuda_dnn.cc:529] Loaded cuDNN version 90300


 14/573 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - loss: 1.5755

...

594/594 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - loss: 0.9401 - val_loss: 0.8959 - learning_rate: 7.0000e-04
Epoch 2/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.6661 - val_loss: 0.8758 - learning_rate: 7.0000e-04
Epoch 3/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.5648 - val_loss: 0.8070 - learning_rate: 7.0000e-04
Epoch 4/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.4757 - val_loss: 0.9458 - learning_rate: 7.0000e-04
Epoch 5/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.4031 - val_loss: 1.0005 - learning_rate: 7.0000e-04
Epoch 6/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.3368 - val_loss: 1.1489 - learning_rate: 7.0000e-04
Epoch 7/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.2866 - val_loss: 1.0463 - learning_rate: 7.0000e-04
Epoch 8/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.2414 - val_loss: 1.0612 - learning_rate: 7.0000e-04
Epoch 9/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.0868 - val_loss: 1.

2026-02-15 16:54:53.609282: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_9', 36 bytes spill stores, 36 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_8', 36 bytes spill stores, 36 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_5', 36 bytes spill stores, 36 bytes spill loads



594/594 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 1.1310

2026-02-15 16:55:15.625895: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_5', 36 bytes spill stores, 36 bytes spill loads



594/594 ━━━━━━━━━━━━━━━━━━━━ 38s 41ms/step - loss: 0.9507 - val_loss: 0.8676 - learning_rate: 7.0000e-04
Epoch 2/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.6682 - val_loss: 0.7860 - learning_rate: 7.0000e-04
Epoch 3/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.5567 - val_loss: 0.9191 - learning_rate: 7.0000e-04
Epoch 4/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.4685 - val_loss: 0.8284 - learning_rate: 7.0000e-04
Epoch 5/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.3992 - val_loss: 0.9121 - learning_rate: 7.0000e-04
Epoch 6/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.3447 - val_loss: 0.9447 - learning_rate: 7.0000e-04
Epoch 7/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 0.2850 - val_loss: 1.0154 - learning_rate: 7.0000e-04
Epoch 8/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.1301 - val_loss: 1.2687 - learning_rate: 3.5000e-04
Epoch 9/30
594/594 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - loss: 0.0812 - val_loss: 1.

## Results analysis

In [ ]:
results_list = []
for k, v in cache.items():
    row = {
        "Configuration": k,
        "Lags": v["lags_count"],
        "MAE": v["metrics_avg"]["MAE"],
        "RMSE": v["metrics_avg"]["RMSE"],
        "Time(s)": v["avg_fit_time"]
    }
    results_list.append(row)

df_res = pd.DataFrame(results_list)
if not df_res.empty:
    df_res = df_res.sort_values("MAE")
    display(df_res)
    
    df_res.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
else:
    print("No results to display.")